# Geospatial centrality demo

Finds the most central educational center within each ownership group in the Comunidad de Madrid dataset, compares two definitions of "most central," and visualizes the result. See [`README.md`](../README.md) for the full methodology and benchmark numbers.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from src.io import load_centers
from src.spark_jobs import most_central_by_centroid, most_central_by_medoid
from src.spark_session import build_spark_session

spark = build_spark_session("demo-notebook")
centers = load_centers(spark, "../data/centros_educativos_madrid.json")
centers.count()

## Most central center per ownership group

Using UTM-projected coordinates (the accurate, primary method -- see README) under both definitions of centrality.

In [ ]:
centroid_result = most_central_by_centroid(
    centers, "ownership", "utm_x", "utm_y", method="utm", implementation="native"
).toPandas()
medoid_result = most_central_by_medoid(centers, "ownership", "utm_x", "utm_y", "utm").toPandas()

centroid_result[["ownership", "center_name", "longitude", "latitude", "distance"]]

In [ ]:
medoid_result[["ownership", "center_name", "longitude", "latitude"]]

The two definitions disagree on every group -- by roughly 1.2-1.6 km each time (see README for the full table). That gap is the whole reason to pick a definition deliberately instead of defaulting to whichever is easiest to compute.

## Map: all centers, with both "most central" picks highlighted

In [ ]:
all_centers = centers.select("ownership", "longitude", "latitude").toPandas()

fig, ax = plt.subplots(figsize=(9, 9))

groups = sorted(all_centers["ownership"].unique())
colors = plt.cm.tab10.colors
for i, group in enumerate(groups):
    subset = all_centers[all_centers["ownership"] == group]
    ax.scatter(
        subset["longitude"], subset["latitude"],
        s=6, alpha=0.35, color=colors[i % len(colors)], label=f"{group} (n={len(subset)})",
    )

ax.scatter(
    centroid_result["longitude"], centroid_result["latitude"],
    marker="*", s=350, color="black", edgecolor="white", linewidth=1, label="nearest-to-centroid pick", zorder=5,
)
ax.scatter(
    medoid_result["longitude"], medoid_result["latitude"],
    marker="D", s=140, color="red", edgecolor="white", linewidth=1, label="medoid pick", zorder=5,
)

# Madrid sits at ~40.4N; scale the longitude axis by 1/cos(lat) so equal
# plot distances represent equal ground distances (see README).
import math
ax.set_aspect(1 / math.cos(math.radians(40.4)))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Educational centers in the Comunidad de Madrid, by ownership")
ax.legend(loc="upper left", fontsize=8, markerscale=1.2)
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
spark.stop()